# Lab 03-02 — Chroma: the persistent vector store

**Track 03 · Vector databases** — lab 01's FAISS index is pure RAM: build it, query it, and when the process exits the index is gone. Chroma takes the opposite trade: everything is written to a directory on disk (`persist_directory`), so the store survives the process that created it.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, faiss, and chroma directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder, the FAISS baseline, the persistent Chroma store, and the reopen story all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

The pipeline, drawn inline:

```
passages.parquet -> HuggingFaceEmbeddings (BGE, local) -> FAISS (lab-01 baseline)
                                                       -> Chroma.from_documents(persist_directory=...)
   -> similarity_search_by_vector_with_relevance_scores -> top-3 per question
   -> close, reopen (in-process AND in a brand-new Python process) -> same answers
```

This lab builds a Chroma store over the same deterministic `Data/corpus/rag-mini-wikipedia` subset as lab 01 and proves persistence the hard way — the store is closed, reopened **in a brand-new Python process**, and answered the same question with the same passage and the same score.

Three things to notice:

* **ON-DISK LAYOUT** — a persistent Chroma collection is a directory containing `chroma.sqlite3` (metadata + collection registry) plus a per-collection folder (the HNSW index). The lab prints the files and their sizes so "persistent" is concrete, not a claim.
* **SCORE CONVENTION** — Chroma's default collection uses the l2 space, and this version of langchain-chroma returns the raw (squared) L2 distance — the same numbers FAISS reports (lab 01). The lab queries FAISS and Chroma with the *same precomputed vectors* and prints the scores side by side: they match to the last few digits. Qdrant (lab 03) is the odd one out with its cosine score.
* **REOPENING** — the store is re-opened by constructing a fresh `Chroma` over the same directory, not by re-adding documents. Re-adding would UPSERT into the existing collection and quietly duplicate every passage — the classic persistent-store footgun.

Persistence is a spectrum, not a checkbox: FAISS trades it away entirely, Chroma pays a little disk I/O for it, and Qdrant (lab 03) offers both a persistent mode and an in-memory one on the same API.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-chroma`, `sentence-transformers`, `faiss-cpu`, and `pandas`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu chromadb langchain-chroma pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import gc
import json
import os
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd

# LangChain + sentence-transformers + faiss + chroma — the only libraries
# this notebook needs. Nothing is imported from the repo's src/ component
# library.
from langchain_chroma import Chroma  # noqa: E402
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# Several scratch notebooks may run in parallel on this machine; keep the
# BLAS thread pool small so embedding does not thrash memory.
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 100` takes the same deterministic head of the 3200-passage corpus as lab 01; `QUESTION_IDS = [1606, 1610]` are real questions whose answers live inside the subset; `TOP_K = 3` is the per-question hit list; `SCORE_TOL = 1e-3` allows last-digit jitter between FAISS and Chroma (separate float32 code paths); and `COLLECTION = "lab02"` names the persistent collection. `PREVIEW` truncates the passage previews the demo prints next to each hit.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus
QUESTION_IDS = [1606, 1610]  # real questions; answers live inside the subset
TOP_K = 3
PREVIEW = 62
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DIM = 768
SCORE_TOL = 1e-3  # FAISS vs Chroma are separate float32 code paths; allow last-digit jitter
COLLECTION = "lab02"


## 2. Load — corpus + questions (same helpers as lab 01)

`load_passages` reads the first `n` rows of `passages.parquet` and returns `(passage_texts, passage_ids)` — the ids are the parquet row indices. `load_questions` pulls the requested `test.parquet` rows as `(question_id, question_text)` pairs, and `preview` flattens a passage onto one line for printing.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions (same helpers as lab 01)
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3. Experiment — embed once, feed the same vectors to FAISS and Chroma, then prove Chroma persists across a process boundary

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model (`normalize_embeddings=True`); we embed the 100 passages once and hand BOTH stores the same vectors through a tiny precomputed passthrough, so the FAISS-vs-Chroma score comparison is apples to apples. FAISS is the lab-01 baseline (`similarity_search_with_score_by_vector`); Chroma is built with `Chroma.from_documents(..., persist_directory=...)` and queried with `similarity_search_by_vector_with_relevance_scores` — the same raw squared-L2 numbers FAISS reports. The on-disk layout is listed, then the store is reopened twice: in-process (fresh `Chroma` over the same directory, no re-add) and in a brand-new Python interpreter that reads the same directory and answers Q1610. This is the same mechanism the shared `src/vectordb/chroma.py` class wraps.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed once, feed the same vectors to FAISS and Chroma,
#    then prove Chroma persists across a process boundary
# --------------------------------------------------------------------------
class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS and Chroma call embed_documents once with the full list; the lookup
    keeps the embed step and the index step separately timed, and would stay
    correct if a store batched the call.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]]):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        raise NotImplementedError("precomputed embeddings cannot embed queries")


def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]

    # --- Embed the subset once; BOTH stores index the same vectors ----------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0
    query_vecs = [embedder.embed_query(q) for _, q in questions]

    # --- FAISS (in-memory, lab 01) — score cross-check baseline ------------
    faiss_store = FAISS.from_documents(
        chunks, embedding=_PrecomputedEmbeddings(passage_texts, passage_vecs)
    )
    faiss_scored = [
        faiss_store.similarity_search_with_score_by_vector(q, k=TOP_K)
        for q in query_vecs
    ]

    # --- Chroma (persistent) — added once, then reopened twice --------------
    tmp = tempfile.TemporaryDirectory(prefix="lab02_chroma_")
    persist_dir = tmp.name

    t0 = time.perf_counter()
    chroma_store = Chroma.from_documents(
        chunks,
        embedding=_PrecomputedEmbeddings(passage_texts, passage_vecs),
        collection_name=COLLECTION,
        persist_directory=persist_dir,
    )
    add_s = time.perf_counter() - t0
    chroma_scored = [
        chroma_store.similarity_search_by_vector_with_relevance_scores(q, k=TOP_K)
        for q in query_vecs
    ]

    # On-disk artifact listing (proves the data lives outside RAM).
    disk_files = sorted(os.listdir(persist_dir))
    disk_bytes = {
        name: os.path.getsize(os.path.join(persist_dir, name)) for name in disk_files
    }

    # In-process reopen: drop the first instance, open a fresh one, no re-add.
    del chroma_store
    gc.collect()
    reopened = Chroma(collection_name=COLLECTION, persist_directory=persist_dir)
    reopened_scored = [
        reopened.similarity_search_by_vector_with_relevance_scores(q, k=TOP_K)
        for q in query_vecs
    ]

    # Cross-process reopen: a brand-new Python interpreter reads the same dir.
    qvec_file = os.path.join(persist_dir, "query_vector.json")
    with open(qvec_file, "w") as fh:
        json.dump(query_vecs[1], fh)  # Q1610 "Who founded Montevideo?"
    sub_script = (
        "import json\n"
        "from langchain_chroma import Chroma\n"
        f"q = json.load(open({qvec_file!r}))\n"
        f"s = Chroma(collection_name={COLLECTION!r}, persist_directory={persist_dir!r})\n"
        "doc, score = s.similarity_search_by_vector_with_relevance_scores(q, k=1)[0]\n"
        'print(f"{doc.metadata.get(chr(105)+chr(100), chr(63))}|{score:.4f}")\n'
    )
    t0 = time.perf_counter()
    proc = subprocess.run(
        [sys.executable, "-c", sub_script], capture_output=True, text=True, timeout=120
    )
    reopen_s = time.perf_counter() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"reopen subprocess failed: {proc.stderr[-500:]}")
    sub_pid_str, sub_score_str = proc.stdout.strip().split("|")

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "embed_s": embed_s,
        "add_s": add_s,
        "reopen_s": reopen_s,
        "faiss_scored": faiss_scored,
        "chroma_scored": chroma_scored,
        "reopened_scored": reopened_scored,
        "disk_files": disk_files,
        "disk_bytes": disk_bytes,
        "sub_pid": int(sub_pid_str),
        "sub_score": float(sub_score_str),
        "dim": len(passage_vecs[0]),
        "indexed": len(passage_vecs),
        "persist_dir": persist_dir,
        "tmp": tmp,
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from five angles: the corpus subset with the embedding timing; the Chroma index build with the on-disk layout (file names + sizes); the top-3 per question with FAISS and Chroma scores side by side (they should match to the last few digits); the persistence proof — same top-1 after an in-process reopen and after a brand-new Python process reopened the same directory; then a takeaway explaining what Chroma trades for persistence and why reopening must never re-add.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 02 — Chroma: the persistent vector store")
    print(f"{BGE_MODEL_NAME} | l2 space | persist_dir on disk")
    print("=" * 66)

    print(f"\n[1] Corpus + embedding:")
    print(f"    {exp['indexed']} passages, dim {exp['dim']}, embedded in {exp['embed_s']:.2f}s")
    print(f"    same vectors indexed into FAISS (lab 01 baseline) AND Chroma")

    print(f"\n[2] Chroma index build:")
    print(f"    {exp['indexed']} passages added in {exp['add_s']:.3f}s")
    print("    on-disk layout of persist_dir:")
    for name, size in exp["disk_bytes"].items():
        print(f"      {name:<45} {size:>9,} B")
    print(f"      {'total':<45} {sum(exp['disk_bytes'].values()):>9,} B")

    print(f"\n[3] Top-{TOP_K} per question — FAISS vs Chroma (squared L2, LOWER = better):")
    for i, (qid, qtext) in enumerate(exp["questions"]):
        print(f'\n    Q[{qid}] "{qtext}"')
        for (fdoc, fscore), (cdoc, cscore) in zip(
            exp["faiss_scored"][i], exp["chroma_scored"][i]
        ):
            match = "SAME" if fdoc.metadata["id"] == cdoc.metadata["id"] else "DIFF"
            print(f"      faiss {fscore:8.4f}  chroma {cscore:8.4f}  "
                  f"[passage {cdoc.metadata['id']}] {preview(cdoc.page_content)}  {match}")

    print(f"\n[4] Persistence — close, reopen, ask again:")
    print("    same top-1 after in-process reopen:")
    for (qid, _), hits in zip(exp["questions"], exp["reopened_scored"]):
        doc, score = hits[0]
        print(f"      Q[{qid}] -> [passage {doc.metadata['id']}] score {score:.4f}")
    print(f"    brand-new Python process reopened the same dir in {exp['reopen_s']:.2f}s:")
    print(f"      Q[{exp['questions'][1][0]}] -> [passage {exp['sub_pid']}] "
          f"score {exp['sub_score']:.4f}")
    print("    the first Chroma instance was deleted; the data survived because")
    print("    it lives in chroma.sqlite3 + the HNSW files, not in RAM.")

    print("\n[5] Takeaway")
    print("    Chroma swaps lab 01's 'index vanishes at exit' for a directory")
    print("    on disk. The price is visible in [2]: building the index touches")
    print("    sqlite and the HNSW files, so add() is slower than FAISS's.")
    print("    And reopening must construct a fresh Chroma over the same dir,")
    print("    never add() again — re-adding upserts and duplicates every")
    print("    passage in the existing collection.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: embedding dimension 768, exactly `N_PASSAGES` passages indexed, FAISS and Chroma return the same passages in the same order with scores agreeing within `SCORE_TOL`, `chroma.sqlite3` exists on disk, the two content checks (Q1610's top-1 names the Spanish founder of Montevideo; Q1606's top-1 mentions Montevideo), the in-process reopen reproduces every top-1 passage id, and the brand-new-process reopen matches the original Q1610 top-1 id and score. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == BGE_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Chroma's score convention matches FAISS's (both squared L2, lower = better).
    all_same_rank = True
    all_scores_close = True
    for fhits, chits in zip(exp["faiss_scored"], exp["chroma_scored"]):
        for (fdoc, fscore), (cdoc, cscore) in zip(fhits, chits):
            all_same_rank &= fdoc.metadata["id"] == cdoc.metadata["id"]
            all_scores_close &= abs(fscore - cscore) < SCORE_TOL
    checks.append(("FAISS and Chroma return the same passages in the same order", all_same_rank))
    checks.append(("FAISS and Chroma scores agree within tolerance", all_scores_close))

    # The store actually wrote to disk.
    checks.append(("persist_dir contains chroma.sqlite3", "chroma.sqlite3" in exp["disk_files"]))

    # Content checks (same as lab 01 — the top-1 answer passages).
    q1610_top = exp["chroma_scored"][1][0][0].page_content.lower()
    checks.append(("Q1610 top-1 names the Spanish founder of Montevideo", "spanish" in q1610_top))
    q1606_top = exp["chroma_scored"][0][0][0].page_content.lower()
    checks.append(("Q1606 top-1 mentions Montevideo", "montevideo" in q1606_top))

    # In-process reopen reproduces the original ranking exactly.
    reopen_matches = all(
        o[0][0].metadata["id"] == r[0][0].metadata["id"]
        for o, r in zip(exp["chroma_scored"], exp["reopened_scored"])
    )
    checks.append(("in-process reopen reproduces every top-1 passage id", reopen_matches))

    # Cross-process reopen answers Q1610 with the same passage and score.
    orig = exp["chroma_scored"][1][0]
    checks.append(("new-process reopen matches the original Q1610 top-1 id",
                   exp["sub_pid"] == orig[0].metadata["id"]))
    checks.append(("new-process reopen matches the original Q1610 score",
                   abs(exp["sub_score"] - orig[1]) < SCORE_TOL))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A few minutes of embedding + index build on rag-mini-wikipedia, plus one short subprocess that reopens the Chroma directory — no downloads, no API calls. `exp` holds everything the demo and gate need. The temporary persist directory is removed automatically when this kernel exits.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The on-disk layout, the FAISS-vs-Chroma score cross-check, and the persistence proof: same answers after an in-process reopen and after a brand-new Python process reopened the same directory.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact.


In [ ]:
verify_gate(exp)
